#XGBoost Preloaded Graph, Meta - Custom Join Data

In [0]:
# Dependencies
import sys
import pandas as pd

# Spark Session (Databricks provides this automatically, but we'll create it explicitly for compatibility)
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()



# Load modules from our Databricks repo
import importlib.util
cv_path = "/Workspace/Shared/Team 4_2/flight-departure-delay-predictive-modeling/notebooks/Cross Validator/cv.py"
spec = importlib.util.spec_from_file_location("cv", cv_path)
cv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cv)


from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, SQLTransformer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBRegressor

In [0]:
data_loader = cv.FlightDelayDataLoader(source="CUSTOM", suffix="_with_graph_and_metamodels")
data_loader.load()  # Must call load() manually when providing dataloader

In [0]:
# ============================================================================
# CATEGORICAL FEATURES
# ============================================================================
categorical_features = [
    # Core flight identifiers
    'op_carrier',              # Operating carrier (AA, UA, DL, etc.)
    'origin',                  # Origin airport code
    'origin_state_abr',        # Origin state abbreviation
    'dest',                    # Destination airport code
    'dest_state_abr',          # Destination state abbreviation
    
    # Temporal categorical features
    'month',                   # Month (1-12) - cyclical patterns
    'dep_time_blk',            # Scheduled departure time block (hour of day)
    'arr_time_blk',            # Scheduled arrival time block (hour of day)
    
    # Previous flight categorical features
    'prev_flight_origin',      # Previous flight origin airport
    'prev_flight_dest',        # Previous flight destination airport
    'prev_flight_op_carrier',  # Previous flight carrier
    
    # Lineage sequence features
    # 'lineage_is_jump',         # Boolean: aircraft jumped airports (prev_dest ≠ curr_origin) - issue: Bool type 
]

# ============================================================================
# NUMERICAL FEATURES - COMPREHENSIVE LIST
# ============================================================================
numerical_features = [
    # ============================================================================
    # Core Flight Characteristics
    # ============================================================================
    'crs_elapsed_time',        # Scheduled elapsed time (air time)
    'distance',                # Flight distance (miles)
    'elevation',               # Airport elevation (if available)
    
    # ============================================================================
    # Flight Lineage - Sequence & Rank
    # ============================================================================
    'lineage_rank',            # Rank of flight in aircraft's sequence (1 = first flight)
    'lineage_num_previous_flights',  # Number of previous flights in rotation
    
    # ============================================================================
    # Flight Lineage - Scheduled Times (Data Leakage Free, Critical for Fundamental Theory)
    # ============================================================================
    'scheduled_lineage_rotation_time_minutes',      # AVAILABLE TIME: Scheduled rotation time (prev_crs_dep → curr_crs_dep)
    'scheduled_lineage_turnover_time_minutes',      # Scheduled turnover time (prev_crs_arr → curr_crs_dep)
    'prev_flight_scheduled_flight_time_minutes',    # Scheduled flight time for previous flight
    'prev_flight_crs_elapsed_time',                 # Scheduled elapsed time for previous flight (backup)
    
    # ============================================================================
    # Flight Lineage - Safe Features (Data Leakage Free, Intelligent Handling)
    # ============================================================================
    'safe_lineage_rotation_time_minutes',           # AVAILABLE TIME (Safe): Safe rotation time (handles data leakage)
    'safe_prev_departure_delay',                    # Safe previous flight departure delay
    'safe_prev_arrival_delay',                      # Safe previous flight arrival delay
    'safe_time_since_prev_arrival',                 # Time since previous flight arrived
    'safe_required_time_prev_flight_minutes',       # Safe required time (air + turnover)
    
    # ============================================================================
    # Flight Lineage - Previous Flight Characteristics (if available at prediction time)
    # ============================================================================
    'prev_flight_distance',                         # Previous flight distance    
        
    # ============================================================================
    # Meta-Model Predictions (Pre-computed, HIGH VALUE!)
    # ============================================================================
    # 'predicted_prev_flight_air_time_XGB_1',         # Predicted previous flight air time
    # 'predicted_prev_flight_turnover_time_XGB_1',    # Predicted previous flight turnover time
    # 'predicted_prev_flight_total_duration_XGB_1',   # Predicted previous flight rotation time
    
    # ============================================================================
    # Current Origin Weather Variables
    # ============================================================================
    'hourlyprecipitation',
    'hourlysealevelpressure',
    'hourlyaltimetersetting',
    'hourlywetbulbtemperature',
    'hourlystationpressure',
    'hourlywinddirection',
    'hourlyrelativehumidity',
    'hourlywindspeed',
    'hourlydewpointtemperature',
    'hourlydrybulbtemperature',
    'hourlyvisibility',
    
    # ============================================================================
    # Previous Flight Weather Variables (OTPW dataset only - HIGH VALUE!)
    # ============================================================================
    # # Hourly weather at previous flight's origin/destination
    # 'prev_flight_hourlyprecipitation',
    # 'prev_flight_hourlywindspeed',
    # 'prev_flight_hourlywinddirection',
    # 'prev_flight_hourlyvisibility',
    # 'prev_flight_hourlydrybulbtemperature',
    # 'prev_flight_hourlydewpointtemperature',
    # 'prev_flight_hourlyrelativehumidity',
    # 'prev_flight_hourlysealevelpressure',
    # 'prev_flight_hourlystationpressure',
    # 'prev_flight_hourlyaltimetersetting',
    # 'prev_flight_hourlywetbulbtemperature',
    # 'prev_flight_hourlywindgustspeed',
    # 'prev_flight_hourlypressurechange',
    # 'prev_flight_hourlypressuretendency',
    
    # # Daily weather at previous flight's origin/destination
    # 'prev_flight_dailyprecipitation',
    # 'prev_flight_dailyaveragewindspeed',
    # 'prev_flight_dailypeakwindspeed',
    # 'prev_flight_dailymaximumdrybulbtemperature',
    # 'prev_flight_dailyminimumdrybulbtemperature',
    # 'prev_flight_dailyaveragedrybulbtemperature',
    # 'prev_flight_dailysnowfall',
    # 'prev_flight_dailysnowdepth',
        
    # ============================================================================
    # Graph Features (If available via add_graph_features.py)
    # ============================================================================
    # 'origin_pagerank_weighted',                     # Origin airport importance (weighted)
    # 'origin_pagerank_unweighted',                   # Origin airport importance (unweighted)
    # 'dest_pagerank_weighted',                       # Destination airport importance (weighted)
    # 'dest_pagerank_unweighted',                     # Destination airport importance (unweighted)
    # 'prev_flight_origin_pagerank_weighted',         # Previous flight origin importance (weighted)
    # 'prev_flight_origin_pagerank_unweighted',       # Previous flight origin importance (unweighted)
    # Note: prev_flight_dest_pagerank not included (see graph_features.md for rationale)
]

In [0]:
imputer = Imputer(
    inputCols=numerical_features,
    outputCols=[f"{col}_IMPUTED" for col in numerical_features],
    strategy="mean"
)

indexer = StringIndexer(
    inputCols=categorical_features,
    outputCols=[f"{col}_INDEX" for col in categorical_features],
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[f"{col}_INDEX" for col in categorical_features],
    outputCols=[f"{col}_VEC" for col in categorical_features]
)

assembler = VectorAssembler(
    inputCols=[f"{col}_VEC" for col in categorical_features] + 
              [f"{col}_IMPUTED" for col in numerical_features],
    outputCol="features",
    handleInvalid="skip"
)

# StandardScaler (required for LinearRegression)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

xgb_regressor = SparkXGBRegressor(num_workers=sc.defaultParallelism, label_col="DEP_DELAY", missing=0.0)

xgb_pipe = Pipeline(stages=[imputer, indexer, encoder, assembler, xgb_regressor])


In [0]:
# trying preloaded graph, meta features
cv_xgb_preloaded_meta_60M_super_safe = cv.FlightDelayCV(
    dataloader = data_loader,
    estimator=xgb_pipe,
    version="60M"
)


In [0]:
cv_xgb_preloaded_meta_60M_super_safe.fit()

In [0]:
cv_xgb_preloaded_meta_60M_super_safe.evaluate()

In [0]:
# Or get both at once
all_results = cv_xgb_preloaded_meta_60M_super_safe.get_results()
fit_df = all_results["fit_results"]
test_df = all_results["evaluate_results"]
test_df